In [ ]:
 !pip -q install faiss-cpu sentence-transformers pypdf transformers accelerate bitsandbytes pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.6 MB/s eta 0:00:00


In [1]:

from preprocessing import *
from RAG_utils import *
from llm_utils import *
from evaluation import *

# To run the grid search function that iterates and evaluates over different combinations

In [2]:


# Put your PDFs/TXTs in this folder:
DOCS_DIR = "/content/drive/MyDrive/Data-Store"

# 3) Creating gold eval set
QA_PATH = "/content/qa_set.jsonl"
write_gold_set(QA_PATH)
###############################################################

llm_models = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
]
embed_models = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
]
k_values = [3, 6]
retriever_types = ["dense"] #as of now

df_grid_summary, df_grid_details = run_grid_configs(
    docs_dir=DOCS_DIR,
    qa_path=QA_PATH,
    llm_models=llm_models,
    embed_models=embed_models,
    k_values=k_values,
    retriever_types=retriever_types,
    use_4bit=True,
    min_score=0.25 #guardrail
)

df_grid_summary

Wrote: /content/qa_set.jsonl


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[build] load=10.64s | chunk=0.12s | embed=2.19s | index=0.00s
Docs: 5 | Chunks: 2238 | EmbDim: 384
[build] load=10.22s | chunk=0.12s | embed=7.66s | index=0.00s
Docs: 5 | Chunks: 2238 | EmbDim: 768
----------------- Qwen/Qwen2.5-1.5B-Instruct --------------------


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


----------------- TinyLlama/TinyLlama-1.1B-Chat-v1.0 --------------------


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


,run_name,llm_model,embed_model,retriever_type,k,RetrievalHit@k,RetrievalPrecision@k,AnswerPhraseRecall,ExactMatchRate,AvgLatency_s,AvgTokensTotal
0,Qwen/Qwen2.5-1.5B-Instruct | sentence-transfor...,Qwen/Qwen2.5-1.5B-Instruct,sentence-transformers/all-mpnet-base-v2,dense,6,0.7,0.116667,0.800000,0.0,9.339521,1694.1
1,TinyLlama/TinyLlama-1.1B-Chat-v1.0 | sentence-...,TinyLlama/TinyLlama-1.1B-Chat-v1.0,sentence-transformers/all-mpnet-base-v2,dense,6,0.7,0.116667,0.800000,0.0,5.945879,1851.5
2,Qwen/Qwen2.5-1.5B-Instruct | sentence-transfor...,Qwen/Qwen2.5-1.5B-Instruct,sentence-transformers/all-mpnet-base-v2,dense,3,0.7,0.233333,0.750000,0.0,8.516686,996.6
3,TinyLlama/TinyLlama-1.1B-Chat-v1.0 | sentence-...,TinyLlama/TinyLlama-1.1B-Chat-v1.0,sentence-transformers/all-mpnet-base-v2,dense,3,0.7,0.233333,0.750000,0.0,5.315771,1069.5
4,Qwen/Qwen2.5-1.5B-Instruct | sentence-transfor...,Qwen/Qwen2.5-1.5B-Instruct,sentence-transformers/all-MiniLM-L6-v2,dense,3,0.6,0.200000,0.891667,0.0,10.952997,1038.5
5,TinyLlama/TinyLlama-1.1B-Chat-v1.0 | sentence-...,TinyLlama/TinyLlama-1.1B-Chat-v1.0,sentence-transformers/all-MiniLM-L6-v2,dense,3,0.6,0.200000,0.866667,0.0,4.000195,1033.3
6,Qwen/Qwen2.5-1.5B-Instruct | sentence-transfor...,Qwen/Qwen2.5-1.5B-Instruct,sentence-transformers/all-MiniLM-L6-v2,dense,6,0.6,0.100000,0.866667,0.0,10.361542,1718.6
7,TinyLlama/TinyLlama-1.1B-Chat-v1.0 | sentence-...,TinyLlama/TinyLlama-1.1B-Chat-v1.0,sentence-transformers/all-MiniLM-L6-v2,dense,6,0.6,0.100000,0.866667,0.0,5.464667,1839.5


In [ ]:
df_grid_summary.to_csv("grid_summary.csv", index=False)

In [ ]:
df_grid_details

,run_name,qid,question,k,retrieval_hit,retrieval_precision@k,answer_phrase_recall,answer_exact_match,latency_s,tokens_total,top1_doc,top1_score,pred_answer,gold_answer
0,Qwen/Qwen2.5-1.5B-Instruct | sentence-transfor...,q1_scope_bsa,"Where does the Bharatiya Sakshya Adhiniyam, 20...",3,1,0.333333,1.00,0,13.977823,1243,"Bharatiya_Nyaya_Sanhita,_2023.pdf",0.518955,system\nYou are a careful assistant. Answer st...,It applies to all judicial proceedings in or b...
1,Qwen/Qwen2.5-1.5B-Instruct | sentence-transfor...,q2_commencement_bsa,"How does the Bharatiya Sakshya Adhiniyam, 2023...",3,0,0.000000,1.00,0,13.038458,1239,"Bharatiya_Nyaya_Sanhita,_2023.pdf",0.524613,system\nYou are a careful assistant. Answer st...,It comes into force on a date appointed by the...
2,Qwen/Qwen2.5-1.5B-Instruct | sentence-transfor...,q3_document_definition,"How is a “document” defined, and does it inclu...",3,1,0.333333,1.00,0,13.037357,1016,250882_english_01042024_0.pdf,0.655224,system\nYou are a careful assistant. Answer st...,A document includes electronic/digital records...
3,Qwen/Qwen2.5-1.5B-Instruct | sentence-transfor...,q4_evidence_definition,What is “evidence” under the Bharatiya Sakshya...,3,0,0.000000,0.25,0,11.862516,1114,"Bharatiya_Nagarik_Suraksha_Sanhita,_2023.pdf",0.552141,system\nYou are a careful assistant. Answer st...,Evidence includes oral evidence (statements pe...
4,Qwen/Qwen2.5-1.5B-Instruct | sentence-transfor...,q5_conclusive_proof,What does “conclusive proof” mean in the Bhara...,3,0,0.000000,1.00,0,6.762598,985,250882_english_01042024_0.pdf,0.650956,system\nYou are a careful assistant. Answer st...,If one fact is declared conclusive proof of an...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,TinyLlama/TinyLlama-1.1B-Chat-v1.0 | sentence-...,q6_bnss_audio_video_electronic,"Under BNSS 2023, what does “audio-video electr...",6,1,0.166667,1.00,0,3.490240,1609,"Bharatiya_Nagarik_Suraksha_Sanhita,_2023.pdf",0.458322,<|system|>\nYou are a careful assistant. Answe...,It includes using communication devices for vi...
76,TinyLlama/TinyLlama-1.1B-Chat-v1.0 | sentence-...,q7_bnss_electronic_communication,"Under BNSS 2023, what is “electronic communica...",6,1,0.166667,1.00,0,10.189093,1755,"Bharatiya_Nagarik_Suraksha_Sanhita,_2023.pdf",0.541147,<|system|>\nYou are a careful assistant. Answe...,Electronic communication is transmission of wr...
77,TinyLlama/TinyLlama-1.1B-Chat-v1.0 | sentence-...,q8_bnss_bailable_nonbailable,Define “bailable offence” and “non-bailable of...,6,1,0.166667,1.00,0,5.948305,1912,"Bharatiya_Nagarik_Suraksha_Sanhita,_2023.pdf",0.708016,<|system|>\nYou are a careful assistant. Answe...,Bailable offence: shown as bailable in the Fir...
78,TinyLlama/TinyLlama-1.1B-Chat-v1.0 | sentence-...,q9_goi1935_lists_counts,"As per the notes, what were the three subject ...",6,1,0.166667,1.00,0,2.924807,1586,UNIT III.pdf,0.735129,<|system|>\nYou are a careful assistant. Answe...,"Federal List (59), Provincial List (54), Concu..."


In [ ]:
df_grid_details.to_csv("grid_details.csv", index=False)

# To run a specifc configuration for inference using the RAG system

In [3]:
def make_answer_fn_from_best(
    best_cfg: Dict[str, str],
    docs_dir: str,
    use_4bit: bool = True,
    min_score: float = 0.25,
    chunk_size: int = 900,
    overlap: int = 150,
):
    """
    Returns a function answer(question)->dict using the best config.
    Uses dense retriever baseline (retriever_type="dense").
    """
    if best_cfg["retriever_type"] != "dense":
        raise ValueError(f"Only dense is implemented in this baseline. Got: {best_cfg['retriever_type']}")

    # Build RAG components for the chosen embedding model
    embed_model, index, chunks = build_rag(
        docs_dir=docs_dir,
        embed_model_name=best_cfg["embed_model"],
        chunk_size=chunk_size,
        overlap=overlap,
    )

    # Load chosen LLM
    llm = HFLocalLLM(best_cfg["llm_model"], use_4bit=use_4bit)

    k = int(best_cfg["k"])

    def answer(question: str):
        """
        Answer any question using the chosen (best) config.
        Returns answer + retrieved chunks + timing + tokens.
        """
        return rag_answer(
            embed_model=embed_model,
            index=index,
            chunks=chunks,
            llm=llm,
            question=question,
            k=k,
            min_score=min_score,
        )

    return answer


In [4]:
# #llm_models = [
#     "Qwen/Qwen2.5-1.5B-Instruct",
#     "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
# ]
# embed_models = [
#     "sentence-transformers/all-MiniLM-L6-v2",
#     "sentence-transformers/all-mpnet-base-v2",
# ]
# k_values = [3, 6]
# retriever_types = ["dense"] #as of now

best_cfg =  {
        "llm_model": "Qwen/Qwen2.5-1.5B-Instruct",
        "embed_model": "sentence-transformers/all-MiniLM-L6-v2",
        "retriever_type": "dense",
        "k": 3,
        "run_name": "inference_1",
    }

answer_fn = make_answer_fn_from_best(
    best_cfg=best_cfg,
    docs_dir=DOCS_DIR,
    use_4bit=True,
    min_score=0.25
)

out = answer_fn("What is the meaning of conclusive proof under the Bharatiya Sakshya Adhiniyam?")
print(out["answer"])


[build] load=10.50s | chunk=0.11s | embed=1.91s | index=0.00s
Docs: 5 | Chunks: 2238 | EmbDim: 384


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


system
You are a careful assistant. Answer strictly from CONTEXT.
If the answer is not in the context, say: "I don't know based on the provided documents.".
Cite sources like (doc_id, chunk_id).
user
CONTEXT:
[SOURCE: 250882_english_01042024_0.pdf | chunk:2 | score:0.615]
ment may, by notification in the Official Gazette, appoint. 2.(1) In this Adhiniyam, unless the context otherwise requires,— (a) "Court" includes all Judges and Magistrates, and all persons, except arbitrators, legally authorised to take evidence; (b) "conclusive proof" means when one fact is declared by this Adhiniyam to be conclusive proof of another, the Court shall, on proof of the one fact, regard the other as proved, and shall not allow evidence to be given for the purpose of disproving it; (c) "disproved" in relation to a fact, means when, after considering the matters before it, the Court either believes that it does not exist, or considers its non-existence so probable that a prudent man ought, under the circ